In [0]:
%run ../../02_common_utils/operations

In [0]:
from datetime import datetime

team_name  = "team_lemma"
bronze_db  = f"charles_schwab_retailbrokerage_dev_{team_name}.bronze"
silver_db  = f"charles_schwab_retailbrokerage_dev_{team_name}.silver"
staging_db = f"charles_schwab_retailbrokerage_dev_{team_name}.staging"
run_id     = datetime.now().strftime("%Y%m%d_%H%M%S")

spark.sql("USE CATALOG charles_schwab_retailbrokerage_dev_team_lemma")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [0]:
try:
    run_info_row = spark.sql(f"SELECT _run_id, _batch FROM {bronze_db}.dailymarket WHERE _batch = 'Batch1' LIMIT 1").first()
    carried_run_id = run_info_row[0] if run_info_row else "unknown"
    carried_batch = run_info_row[1] if run_info_row else "Batch1"
except Exception:
    carried_run_id = "unknown"
    carried_batch = "Batch1"

In [0]:
log_pipeline_message(spark, carried_run_id, 'INFO', 'bronze_sliver_market_dailymarket_batch1', f'Starting processing for DailyMarket Batch 1')
start_pipeline_run(spark, carried_run_id, carried_batch)
log_domain_run_status(spark, carried_run_id, carried_batch, 'MARKET', 'RUNNING')

In [0]:
from pyspark.sql.functions import col, trim, to_date, current_timestamp, lit
from pyspark.sql.types import DecimalType, LongType

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

silver_b1 = (
    spark.table(f"{bronze_db}.dailymarket")
    .filter(col("_batch") == "Batch1")
    .select(
        to_date(trim(col("DM_DATE")),   "yyyy-MM-dd").alias("dm_date"),
        trim(col("DM_S_SYMB")).alias("dm_s_symb"),
        trim(col("DM_CLOSE")).cast(DecimalType(8,2)).alias("dm_close"),
        trim(col("DM_HIGH")).cast(DecimalType(8,2)).alias("dm_high"),
        trim(col("DM_LOW")).cast(DecimalType(8,2)).alias("dm_low"),
        trim(col("DM_VOL")).cast(LongType()).alias("dm_vol"),
        col("_batch"),
        lit(run_id).alias("_run_id"),
        current_timestamp().alias("_load_ts"),
    )
)

(silver_b1.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{silver_db}.markethistory"))

print(f"silver.markethistory B1 rows: {spark.table(f'{silver_db}.markethistory').count():,}")


In [0]:
log_domain_run_status(spark, carried_run_id, carried_batch, 'MARKET', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'bronze_sliver_market_dailymarket_batch1', 'Successfully completed processing for DailyMarket Batch 1.')